# W10B 독립 R Colab A · 영향력 (강의 B)

회귀분석의 이해 · 2026-11-04 · 주교재 §5.8–5.15. 이번에는 B·C를 보강하며 §6.1–6.3은 제외합니다. 성적 반영 과제나 제출 기한을 새로 정하지 않습니다.

**예상과 이유 → 손계산 → 실행 → 그림 비교 → 한 문장 해석**을 남깁니다. 설명 예제와 다른 관측을 쓰는 연습을 구별하세요. TODO는 뒤의 설명 예제를 막지 않습니다. 새로 시작하면 준비 셀부터 실행합니다.

[온라인 R](https://lunalab-ai.github.io/regre/w10b.html) · [누적 앱 편집기](https://lunalab-ai.github.io/regre/apps/w10b/edit/index.html) · [입출력과 실제 함수 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w10b/src/diagnostics-w10b-api.md)

`source()`는 정의를 읽고 `lm()`은 계수를 적합합니다. `w10b_deletion_trace(lm,원래행번호)`는 원래 모든 위치의 두 예측표와 지표를 반환합니다. `w10b_effect_trace(lm,"Distance")`는 같은 행의 두 보조회귀 잔차·원래 x·CPR좌표를 반환합니다. 함수가 자료를 자동 삭제하지 않습니다.

자료는 교육용5행(x/y교육용단위), NY20행(Nitrogen원척도/ComIndl%), Hills35행(Time초/Distance마일/Climb고도상승피트)입니다. 준비 코드는 동봉 자료→지정 캐시→고정태그 다운로드 순으로 필요한7파일을 확인하고 MD5를 대조합니다. 편집 파일을 덮어쓰거나 실패를 가짜 자료로 대체하지 않습니다. 공개 ZIP/Release의 SHA-256 검증과 구별합니다.


In [ ]:
# Prefer bundled files; otherwise download from one fixed public tag.
tag <- "2026-fall-w10b"
local_dirs <- c("data","labs/student/w10b/data","../../labs/student/w10b/data",
                "../../../labs/student/w10b/data","../data")
found <- local_dirs[file.exists(file.path(local_dirs,"diagnostics-w10b.R"))]
cache_root <- Sys.getenv("REGRE_CACHE_DIR",unset=file.path(path.expand("~"),".cache","regre"))
data_dir <- if(length(found)) found[1] else file.path(cache_root,tag)
dir.create(data_dir,recursive=TRUE,showWarnings=FALSE)
expected <- c("diagnostics-base-w10b.R"="1d6d198179eb52922e0b7da3a3836838","diagnostics-influence-w10b.R"="106f87499e0a0565f4c19d75b7e26c9a","diagnostics-effects-w10b.R"="df333742f83c7f635427673c62acf214","diagnostics-w10b.R"="bf0d09021d75d1e04dbbbfc64fd23403","hand-five-w10b.csv"="d5911a65bf38d08ef947d43cb7ebf0e9","nyrivers-w10b.csv"="6f67635643fbb5aac0b37b7a8fed0d26","scottish-hills-w10b.csv"="ea6e5b9206884d205cfa393d89e0f84a")
options(timeout=120)
for(name in names(expected)) {
 target <- file.path(data_dir,name)
 if(!file.exists(target)) {
  url <- paste0("https://raw.githubusercontent.com/lunalab-ai/regre/",tag,"/labs/student/w10b/data/",name)
  status <- download.file(url,target,mode="wb")
  if(status!=0) stop("Download failed. Check the connection and rerun preparation.")
  cat(name,": fixed-tag download\n")
 } else cat(name,": bundled/cache file\n")
 if(unname(tools::md5sum(target))!=expected[[name]])
  stop("Unexpected file bytes. Preserve edited files; use a fresh folder or cache.")
}
source(file.path(data_dir,"diagnostics-base-w10b.R"),encoding="UTF-8")
source(file.path(data_dir,"diagnostics-influence-w10b.R"),encoding="UTF-8")
source(file.path(data_dir,"diagnostics-effects-w10b.R"),encoding="UTF-8")
source(file.path(data_dir,"diagnostics-w10b.R"),encoding="UTF-8")
toy <- read.csv(file.path(data_dir,"hand-five-w10b.csv"))
ny <- read.csv(file.path(data_dir,"nyrivers-w10b.csv"))
hills <- read.csv(file.path(data_dir,"scottish-hills-w10b.csv"))
ft <- lm(y~x,toy); fn <- lm(Nitrogen~ComIndl,ny)
fh <- lm(Time~Distance+Climb,hills)
stopifnot(nrow(toy)==5,nrow(ny)==20,nrow(hills)==35,hills$Time[23]==2859)
cat("W10B_DATA_READY: toy5 / NY20 / Hills35; units unchanged\n")


## E01 · 관측과 적합값을 따로 읽기

교육용3번의 관측과 전체 적합값을 비교합니다. 아래 출력은 설명 예제이며 L01은 다른2번으로 풉니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f03.png" alt="단계 그림" width="900">

[원본 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f03.png)

In [ ]:
print(c(observed=toy$y[3],fitted=fitted(ft)[3],residual=residuals(ft)[3],h=hatvalues(ft)[3]))
plot(toy$x,toy$y,pch=19,xlab="x (teaching units)",ylab="y (teaching units)");abline(ft,col="steelblue",lwd=2)


## L01 · 필수 · 관측·잔차·위치

교육용5행에서 2번은 x=1,y=2입니다. 전체 적합식은 $\hat y=1+0.8x$, $\bar x=2$, $S_{xx}=10$, n=5,k=2입니다. 2번의 적합값·잔차·h를 계산하고 잔차의 방향을 그림에서 설명하세요.

힌트: $h=1/n+(x-\bar x)^2/S_{xx}$이며 잔차는 관측−적합입니다.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L01_prediction <- NA_real_
my_L01_residual <- NA_real_
my_L01_h <- NA_real_

## E02–E03 · 제외 후 모든 원래 위치에서 비교

w10b_deletion_trace는 원래 lm과 원래 관측 번호를 받아 실제 재적합·원래 모든 위치의 두 예측 표·지표·규모를 반환합니다. difference는 전체−제외 후, squared_change는 그 제곱입니다. 원자료를 바꾸지 않습니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f09.png" alt="단계 그림" width="900">

[원본 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f09.png)

In [ ]:
d4 <- w10b_deletion_trace(ft,4)
print(d4$predictions,digits=6)
print(c(numerator=d4$numerator,denominator=d4$denominator,Cook=d4$Cook_from_predictions))


## E04 · 자기 위치와 삭제 규모

score의 Cook과 DFFITS는 다른 질문에 답합니다. DFFITS는 전체−제외 후의 부호와 삭제 s를 사용합니다. 원래 fit의 h를 그대로 연결합니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f13.png" alt="단계 그림" width="900">

[원본 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f13.png)

In [ ]:
print(d4$score[,c("row","h","raw","Cook","DFFITS")],digits=6)
print(c(full_s=d4$s,deleted_s=d4$s_deleted))


## L02 · 필수 · 다른 점을 제외해 Cook 만들기

같은 교육용5행에서 이번에는 3번을 제외합니다. 제외 후 직선은 $1.15+0.8x$입니다. 원래 모든5개 위치의 두 예측 차이·제곱·합을 표로 만들고 Cook을 구하세요. 전체 SSE=0.8,n=5,k=2입니다. 제외 후 SSE=0.35,df=2일 때 DFFITS의 부호와 제외 후 자기 예측의 이동 방향도 설명하세요.

힌트: 전체−제외 후를 모든 원래 x에서 계산합니다. 전체 규모와 삭제 규모를 구별하세요.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L02_Cook <- NA_real_
my_L02_DFFITS <- NA_real_
my_L02_reason <- ""

## E05 · 교재 NY의 P-R

같은20행, Nitrogen~ComIndl,k=2입니다. diag_influence는 현재 모형의 원래 각 행을 반환합니다. 아래는 설명용4/5번이며 L03은7번을 직접 계산합니다. 가로R·세로P입니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f20.png" alt="단계 그림" width="900">

[원본 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w10b/course/notion/w10b/images/w10b-f20.png)

In [ ]:
ni <- diag_influence(fn)
print(ni[c(4,5),c("row","h","Cook","DFFITS","potential","residual_component","Hadi")],digits=6)
plot(ni$residual_component,ni$potential,pch=19,xlab="Hadi residual component R",ylab="Potential P");text(ni$residual_component,ni$potential,labels=ni$row,pos=3,cex=.7)


## L03 · 필수 · P-R 한 점과 축 오류 수정

NY 단순모형 n=20,k=2의 7번은 e=0.699492144237,h=0.082605785602,SSE=2.595399158263입니다. d²,P,R,H를 계산하고 좌표를 쓰세요. `plot(P,R,xlab="R",ylab="P")`의 어느 연결을 고쳐야 하는지도 설명하세요.

힌트: P와 R을 먼저 별도로 계산하고, 가로 R·세로 P에 놓습니다.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L03_P <- NA_real_
my_L03_R <- NA_real_
my_L03_H <- NA_real_

## L06 · 필수 · 앱 연결 수정과 저장

누적 앱 편집기에서 point_focus 출력의 Cook 필드를 DFFITS로 바꾸고 UI제목과 출력 문구도 맞추세요. 3번과4번을 선택해 먼저 예상한 부호와 결과를 비교합니다. 원래5행 예측표 CSV와 수정 코드를 저장하고 재시작하세요. 마지막에는 X01–X03 중 한 사례를 코드 없이 설명하세요.

힌트: 입력→selected_trace→selected_score→point_focus의 연결을 찾습니다. 원자료는 바꾸지 않습니다.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L06_prediction <- ""
my_L06_saved_files <- ""
my_L06_restart_observation <- ""

## L07 · 반복 · 단위 변경

교육용5행에서 모든 y를10배하고 같은 x로 다시 적합합니다. 보통잔차,s,h,내적 표준화잔차,Cook이 각각 어떻게 바뀌는지 먼저 예상한 뒤 검산하세요.

힌트: 분자·분모가 같은 단위 변화의 영향을 받는지 확인합니다.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L07_prediction <- ""

## L08 · 반복 · 성분 좌표와 정보 손실

별도 설명용 관측에서 P=0.4,R=0.8입니다. P-R좌표와 H를 쓰고, 잔차 부호를 알 수 있는지 설명하세요. 이 숫자들을 NY의 특정 강 값이라고 해석하지 않습니다.

힌트: 점의 순서는 가로값,세로값입니다.

실행 전 예상·중간 계산·차이·해석을 워크북에 기록하세요.

In [ ]:
my_L08_coordinate <- c(NA_real_,NA_real_)

In [ ]:
stopifnot(nrow(toy)==5,nrow(ny)==20,nrow(hills)==35,
 abs(d4$Cook_from_predictions-cooks.distance(ft)[4])<1e-10,
 max(abs(ni$Cook-cooks.distance(fn)))<1e-9,
 abs(sum(hatvalues(ft))-2)<1e-10)
cat("W10B_INFLUENCE_RUNTIME_CHECK_PASSED\n")